In [ ]:
# ============================================================
# Create batch2 manual-labeling sample
# This does NOT change the manual-labeling tool logic.
# It only creates a new input CSV for the next labeling batch.
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive"

RULE_BASED_RESULTS_PATH = f"{ROOT}/taekwondo_rule_based_classification_results.csv"
OLD_LABELS_PATH = f"{ROOT}/taekwondo_manual_labels.csv"

BATCH_ID = "batch2"
BATCH_SAMPLE_PATH = f"{ROOT}/taekwondo_manual_labeling_{BATCH_ID}_with_video_path.csv"

VIDEO_ROOT = f"{ROOT}/taekwondo_videos"
KEYPOINT_CSV_DIR = f"{ROOT}/taekwondo_keypoints_csv_main_actor"

RANDOM_STATE = 44

VIDEO_EXTS = (
    ".mp4", ".MP4",
    ".mov", ".MOV",
    ".avi", ".AVI",
    ".mkv", ".MKV",
)

TARGET_RULE_ACTIONS = {
    "Axe Kick / High Kick": 30,
    "Side Kick": 30,
    "Front Kick": 30,
    "Roundhouse Kick": 20,
}


def clean_text(x):
    text = str(x).strip()
    if text.lower() in ["nan", "none", "null"]:
        return ""
    return text


def collect_video_files(video_root):
    video_files = []

    for ext in VIDEO_EXTS:
        video_files.extend(
            glob.glob(os.path.join(video_root, "**", f"*{ext}"), recursive=True)
        )

    return sorted(set(video_files))


def build_video_map(video_files):
    video_map = {}

    for p in video_files:
        stem = os.path.splitext(os.path.basename(p))[0]
        video_map[stem] = p

    return video_map


def find_video_path(video_name, video_map):
    video_name = clean_text(video_name).replace("_keypoints", "")

    if video_name in video_map:
        return video_map[video_name]

    for stem, path in video_map.items():
        if video_name == stem:
            return path
        if video_name in stem or stem in video_name:
            return path

    return ""


def repair_csv_path(row):
    old_path = clean_text(row.get("csv_path", ""))

    if os.path.exists(old_path):
        return old_path

    video_name = clean_text(row.get("video_name", "")).replace("_keypoints", "")
    candidate = os.path.join(KEYPOINT_CSV_DIR, f"{video_name}_keypoints.csv")

    if os.path.exists(candidate):
        return candidate

    return old_path


# 1. Load rule-based results
df_rule = pd.read_csv(RULE_BASED_RESULTS_PATH)

print("Rule-based results loaded:", RULE_BASED_RESULTS_PATH)
print("Rule results:", df_rule.shape)
print("\nRule-based distribution:")
print(df_rule["predicted_action"].value_counts(dropna=False))


# 2. Load already labeled videos, so batch2 will not repeat them
already_labeled = set()

if os.path.exists(OLD_LABELS_PATH):
    df_old = pd.read_csv(OLD_LABELS_PATH)
    already_labeled = set(df_old["video_name"].astype(str).str.strip())
    print("\nOld manual labels loaded:", OLD_LABELS_PATH)
    print("Already labeled / already used videos:", len(already_labeled))
else:
    print("\nNo old manual label file found. Batch2 will not exclude previous labels.")


# 3. Match video paths
video_files = collect_video_files(VIDEO_ROOT)
video_map = build_video_map(video_files)

print("\nOriginal videos found:", len(video_files))

df_candidates = df_rule.copy()
df_candidates["video_name"] = df_candidates["video_name"].astype(str).str.strip()

if "original_video_path" not in df_candidates.columns:
    df_candidates["original_video_path"] = ""

df_candidates["original_video_path"] = df_candidates.apply(
    lambda row: (
        row["original_video_path"]
        if isinstance(row.get("original_video_path", ""), str)
        and os.path.exists(row.get("original_video_path", ""))
        else find_video_path(row["video_name"], video_map)
    ),
    axis=1,
)

df_candidates["csv_path"] = df_candidates.apply(repair_csv_path, axis=1)

df_candidates["video_exists"] = df_candidates["original_video_path"].apply(
    lambda p: isinstance(p, str) and os.path.exists(p)
)

df_candidates["csv_exists"] = df_candidates["csv_path"].apply(
    lambda p: isinstance(p, str) and os.path.exists(p)
)


# 4. Filter candidates
df_candidates = df_candidates[
    (~df_candidates["video_name"].isin(already_labeled)) &
    (df_candidates["video_exists"]) &
    (df_candidates["csv_exists"])
].copy()

print("\nAvailable candidates after removing old labels:")
print("Rows:", len(df_candidates))
print(df_candidates["predicted_action"].value_counts(dropna=False))


# 5. Sample batch2
samples = []

for action, n in TARGET_RULE_ACTIONS.items():
    part = df_candidates[df_candidates["predicted_action"] == action].copy()

    sample_n = min(n, len(part))

    print(f"{action}: available={len(part)}, sampled={sample_n}")

    if sample_n > 0:
        samples.append(part.sample(n=sample_n, random_state=RANDOM_STATE))

if not samples:
    raise RuntimeError("No candidates available for batch2.")

df_batch = pd.concat(samples, ignore_index=True)
df_batch = df_batch.drop_duplicates(subset=["video_name"], keep="first").copy()

# 6. Prepare label columns
df_batch["manual_label"] = ""
df_batch["label_status"] = "unlabeled"
df_batch["notes"] = ""
df_batch["batch_id"] = BATCH_ID

front_cols = [
    "batch_id",
    "video_name",
    "original_video_path",
    "csv_path",
    "predicted_action",
    "manual_label",
    "label_status",
    "notes",
]

other_cols = [c for c in df_batch.columns if c not in front_cols]
df_batch = df_batch[front_cols + other_cols]

df_batch.to_csv(BATCH_SAMPLE_PATH, index=False, encoding="utf-8-sig")

print("\nBatch2 sample saved:")
print(BATCH_SAMPLE_PATH)

print("\nBatch2 size:", len(df_batch))
print("\nBatch2 distribution:")
print(df_batch["predicted_action"].value_counts(dropna=False))

display(df_batch.head(20))

Rule-based results loaded: /content/drive/MyDrive/taekwondo_rule_based_classification_results.csv
Rule results: (678, 17)

Rule-based distribution:
predicted_action
Axe Kick / High Kick    200
Front Kick              191
Roundhouse Kick         158
Unknown                  57
Punch / Hand Motion      47
Side Kick                25
Name: count, dtype: int64

Old manual labels loaded: /content/drive/MyDrive/taekwondo_manual_labels.csv
Already labeled / already used videos: 115

Original videos found: 895

Available candidates after removing old labels:
Rows: 563
predicted_action
Axe Kick / High Kick    170
Front Kick              161
Roundhouse Kick         128
Unknown                  57
Punch / Hand Motion      47
Name: count, dtype: int64
Axe Kick / High Kick: available=170, sampled=30
Side Kick: available=0, sampled=0
Front Kick: available=161, sampled=30
Roundhouse Kick: available=128, sampled=20

Batch2 sample saved:
/content/drive/MyDrive/taekwondo_manual_labeling_batch2_with_vide

,batch_id,video_name,original_video_path,csv_path,predicted_action,manual_label,label_status,notes,right_foot_highest,left_foot_highest,...,left_foot_y_range,right_foot_speed,left_foot_speed,max_foot_speed,body_vertical_range,right_wrist_x_range,left_wrist_x_range,max_wrist_x_range,video_exists,csv_exists
0,batch2,V_20240902_104232_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.174299,0.448667,...,0.363592,0.162323,0.019921,0.162323,0.174718,0.089547,0.116533,0.116533,True,True
1,batch2,IMG_5982,/content/drive/MyDrive/taekwondo_videos/junior...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.225038,0.240942,...,0.616681,0.067270,0.024754,0.067270,0.156300,0.365625,0.430340,0.430340,True,True
2,batch2,IMG_5989,/content/drive/MyDrive/taekwondo_videos/junior...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.262810,0.785577,...,0.051709,0.076476,0.005468,0.076476,0.163469,0.087454,0.076546,0.087454,True,True
3,batch2,VID_20240902_103016,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.313122,0.829743,...,0.045021,0.158055,0.011913,0.158055,0.155990,0.120019,0.112735,0.120019,True,True
4,batch2,IMG_6036,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.316810,0.596502,...,0.289915,0.039059,0.010885,0.039059,0.138183,0.183656,0.132863,0.183656,True,True
5,batch2,IMG_4829,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.858780,0.270414,...,0.611298,0.006951,0.159846,0.159846,0.183155,0.189605,0.053823,0.189605,True,True
6,batch2,IMG_5599,/content/drive/MyDrive/taekwondo_videos/junior...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.147990,0.180433,...,0.716482,0.140266,0.105870,0.140266,0.193584,0.256009,0.179430,0.256009,True,True
7,batch2,IMG_5928,/content/drive/MyDrive/taekwondo_videos/junior...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.269798,0.780060,...,0.081391,0.051460,0.006986,0.051460,0.154670,0.208383,0.152772,0.208383,True,True
8,batch2,IMG_4804,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.322434,0.339945,...,1.014814,0.078936,0.014022,0.078936,0.185738,0.588674,0.599008,0.599008,True,True
9,batch2,IMG_4837,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.279997,0.837388,...,0.176627,0.070720,0.016763,0.070720,0.210363,0.304803,0.264991,0.304803,True,True


In [ ]:
# ============================================================
# Create batch3 manual-labeling sample
# Logic unchanged: only creates a new batch CSV.
# ============================================================

import os
import glob
import pandas as pd

ROOT = "/content/drive/MyDrive"

RULE_BASED_RESULTS_PATH = f"{ROOT}/taekwondo_rule_based_classification_results.csv"

# 已經標過的 label 檔都放進來，避免 batch3 抽到重複影片
OLD_LABEL_FILES = [
    f"{ROOT}/taekwondo_manual_labels.csv",
    f"{ROOT}/taekwondo_manual_labels_batch2.csv",
]

BATCH_ID = "batch3"
BATCH_SAMPLE_PATH = f"{ROOT}/taekwondo_manual_labeling_{BATCH_ID}_with_video_path.csv"

VIDEO_ROOT = f"{ROOT}/taekwondo_videos"
KEYPOINT_CSV_DIR = f"{ROOT}/taekwondo_keypoints_csv_main_actor"

RANDOM_STATE = 45

VIDEO_EXTS = (
    ".mp4", ".MP4",
    ".mov", ".MOV",
    ".avi", ".AVI",
    ".mkv", ".MKV",
)

#
TARGET_RULE_ACTIONS = {
    "Axe Kick / High Kick": 60,
    "Side Kick": 60,
    "Front Kick": 60,
    "Roundhouse Kick": 60,
}


def clean_text(x):
    text = str(x).strip()
    if text.lower() in ["nan", "none", "null"]:
        return ""
    return text


def collect_video_files(video_root):
    video_files = []

    for ext in VIDEO_EXTS:
        video_files.extend(
            glob.glob(os.path.join(video_root, "**", f"*{ext}"), recursive=True)
        )

    return sorted(set(video_files))


def build_video_map(video_files):
    video_map = {}

    for p in video_files:
        stem = os.path.splitext(os.path.basename(p))[0]
        video_map[stem] = p

    return video_map


def find_video_path(video_name, video_map):
    video_name = clean_text(video_name).replace("_keypoints", "")

    if video_name in video_map:
        return video_map[video_name]

    for stem, path in video_map.items():
        if video_name == stem:
            return path
        if video_name in stem or stem in video_name:
            return path

    return ""


def repair_csv_path(row):
    old_path = clean_text(row.get("csv_path", ""))

    if os.path.exists(old_path):
        return old_path

    video_name = clean_text(row.get("video_name", "")).replace("_keypoints", "")
    candidate = os.path.join(KEYPOINT_CSV_DIR, f"{video_name}_keypoints.csv")

    if os.path.exists(candidate):
        return candidate

    return old_path


# 1. Load rule-based results
df_rule = pd.read_csv(RULE_BASED_RESULTS_PATH)

print("Rule-based results loaded:", RULE_BASED_RESULTS_PATH)
print("Rule results:", df_rule.shape)
print("\nRule-based distribution:")
print(df_rule["predicted_action"].value_counts(dropna=False))


# 2. Collect already-used videos from batch1 + batch2
already_used = set()

for path in OLD_LABEL_FILES:
    if os.path.exists(path):
        df_old = pd.read_csv(path)
        if "video_name" in df_old.columns:
            names = set(df_old["video_name"].astype(str).str.strip())
            already_used.update(names)
            print(f"Loaded used videos from {path}: {len(names)}")
    else:
        print("Not found, skipped:", path)

print("\nTotal already-used videos:", len(already_used))


# 3. Match original video paths
video_files = collect_video_files(VIDEO_ROOT)
video_map = build_video_map(video_files)

print("\nOriginal videos found:", len(video_files))

df_candidates = df_rule.copy()
df_candidates["video_name"] = df_candidates["video_name"].astype(str).str.strip()

if "original_video_path" not in df_candidates.columns:
    df_candidates["original_video_path"] = ""

df_candidates["original_video_path"] = df_candidates.apply(
    lambda row: (
        row["original_video_path"]
        if isinstance(row.get("original_video_path", ""), str)
        and os.path.exists(row.get("original_video_path", ""))
        else find_video_path(row["video_name"], video_map)
    ),
    axis=1,
)

df_candidates["csv_path"] = df_candidates.apply(repair_csv_path, axis=1)

df_candidates["video_exists"] = df_candidates["original_video_path"].apply(
    lambda p: isinstance(p, str) and os.path.exists(p)
)

df_candidates["csv_exists"] = df_candidates["csv_path"].apply(
    lambda p: isinstance(p, str) and os.path.exists(p)
)


# 4. Filter out old videos and invalid paths
df_candidates = df_candidates[
    (~df_candidates["video_name"].isin(already_used)) &
    (df_candidates["video_exists"]) &
    (df_candidates["csv_exists"])
].copy()

print("\nAvailable candidates after removing old labels:")
print("Rows:", len(df_candidates))
print(df_candidates["predicted_action"].value_counts(dropna=False))


# 5. Sample batch3
samples = []

for action, n in TARGET_RULE_ACTIONS.items():
    part = df_candidates[df_candidates["predicted_action"] == action].copy()

    sample_n = min(n, len(part))

    print(f"{action}: available={len(part)}, sampled={sample_n}")

    if sample_n > 0:
        samples.append(part.sample(n=sample_n, random_state=RANDOM_STATE))

if not samples:
    raise RuntimeError("No candidates available for batch3.")

df_batch = pd.concat(samples, ignore_index=True)
df_batch = df_batch.drop_duplicates(subset=["video_name"], keep="first").copy()

# 6. Prepare label columns
df_batch["manual_label"] = ""
df_batch["label_status"] = "unlabeled"
df_batch["notes"] = ""
df_batch["batch_id"] = BATCH_ID

front_cols = [
    "batch_id",
    "video_name",
    "original_video_path",
    "csv_path",
    "predicted_action",
    "manual_label",
    "label_status",
    "notes",
]

other_cols = [c for c in df_batch.columns if c not in front_cols]
df_batch = df_batch[front_cols + other_cols]

df_batch.to_csv(BATCH_SAMPLE_PATH, index=False, encoding="utf-8-sig")

print("\nBatch3 sample saved:")
print(BATCH_SAMPLE_PATH)

print("\nBatch3 size:", len(df_batch))
print("\nBatch3 distribution:")
print(df_batch["predicted_action"].value_counts(dropna=False))

display(df_batch.head(20))

Rule-based results loaded: /content/drive/MyDrive/taekwondo_rule_based_classification_results.csv
Rule results: (678, 17)

Rule-based distribution:
predicted_action
Axe Kick / High Kick    200
Front Kick              191
Roundhouse Kick         158
Unknown                  57
Punch / Hand Motion      47
Side Kick                25
Name: count, dtype: int64
Loaded used videos from /content/drive/MyDrive/taekwondo_manual_labels.csv: 115
Loaded used videos from /content/drive/MyDrive/taekwondo_manual_labels_batch2.csv: 80

Total already-used videos: 195

Original videos found: 895

Available candidates after removing old labels:
Rows: 483
predicted_action
Axe Kick / High Kick    140
Front Kick              131
Roundhouse Kick         108
Unknown                  57
Punch / Hand Motion      47
Name: count, dtype: int64
Axe Kick / High Kick: available=140, sampled=60
Side Kick: available=0, sampled=0
Front Kick: available=131, sampled=60
Roundhouse Kick: available=108, sampled=60

Batch3 sa

,batch_id,video_name,original_video_path,csv_path,predicted_action,manual_label,label_status,notes,right_foot_highest,left_foot_highest,...,left_foot_y_range,right_foot_speed,left_foot_speed,max_foot_speed,body_vertical_range,right_wrist_x_range,left_wrist_x_range,max_wrist_x_range,video_exists,csv_exists
0,batch3,V_20240902_110137_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.372348,0.254696,...,0.728039,0.047715,0.098170,0.098170,0.151617,0.235034,0.181100,0.235034,True,True
1,batch3,IMG_6028,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.300330,0.587960,...,0.310552,0.038032,0.008610,0.038032,0.136585,0.119045,0.119722,0.119722,True,True
2,batch3,IMG_6902,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.393327,0.334197,...,0.527322,0.015676,0.080951,0.080951,0.132400,0.237048,0.341564,0.341564,True,True
3,batch3,IMG_6861,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.768762,0.261563,...,0.540487,0.003957,0.066280,0.066280,0.160899,0.095614,0.084678,0.095614,True,True
4,batch3,IMG_6253,/content/drive/MyDrive/taekwondo_videos/junior...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.241001,0.455273,...,0.385982,0.104378,0.028572,0.104378,0.153721,0.209033,0.166302,0.209033,True,True
5,batch3,IMG_6700,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.303185,0.636195,...,0.297877,0.053833,0.010018,0.053833,0.169014,0.246622,0.189350,0.246622,True,True
6,batch3,IMG_5535,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.214334,0.453397,...,0.483721,0.098099,0.021196,0.098099,0.192696,0.281250,0.288935,0.288935,True,True
7,batch3,V_20240902_110613_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.231215,0.376515,...,0.412725,0.090179,0.045675,0.090179,0.147928,0.272728,0.254206,0.272728,True,True
8,batch3,video_533886521410584754-IppVxgSp,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.217752,0.281661,...,0.493244,0.073504,0.058371,0.073504,0.153142,0.182819,0.122783,0.182819,True,True
9,batch3,IMG_5916,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,,unlabeled,,0.323450,0.796857,...,0.145881,0.043695,0.008364,0.043695,0.150257,0.201599,0.196014,0.201599,True,True


In [ ]:
# ============================================================
# Create batch5 manual-labeling sample
# Logic unchanged: only creates a new batch CSV.
# ============================================================

import os
import glob
import pandas as pd

ROOT = "/content/drive/MyDrive"

RULE_BASED_RESULTS_PATH = f"{ROOT}/taekwondo_rule_based_classification_results.csv"

OLD_LABEL_FILES = [
    f"{ROOT}/taekwondo_manual_labels.csv",
    f"{ROOT}/taekwondo_manual_labels_batch2.csv",
    f"{ROOT}/taekwondo_manual_labels_batch3.csv",
    f"{ROOT}/taekwondo_manual_labels_batch4.csv",
]

BATCH_ID = "batch5"
BATCH_SAMPLE_PATH = f"{ROOT}/taekwondo_manual_labeling_{BATCH_ID}_with_video_path.csv"

VIDEO_ROOT = f"{ROOT}/taekwondo_videos"
KEYPOINT_CSV_DIR = f"{ROOT}/taekwondo_keypoints_csv_main_actor"

RANDOM_STATE = 46

VIDEO_EXTS = (
    ".mp4", ".MP4",
    ".mov", ".MOV",
    ".avi", ".AVI",
    ".mkv", ".MKV",
)

# 第四批一次多一點：最多 200 支候選
# 超過 20 秒仍會由原本 manual labeling tool 自動 exclude
TARGET_RULE_ACTIONS = {
    "Axe Kick / High Kick": 50,
    "Side Kick": 50,
    "Front Kick": 50,
    "Roundhouse Kick": 50,
}


def clean_text(x):
    text = str(x).strip()
    if text.lower() in ["nan", "none", "null"]:
        return ""
    return text


def collect_video_files(video_root):
    video_files = []

    for ext in VIDEO_EXTS:
        video_files.extend(
            glob.glob(os.path.join(video_root, "**", f"*{ext}"), recursive=True)
        )

    return sorted(set(video_files))


def build_video_map(video_files):
    video_map = {}

    for p in video_files:
        stem = os.path.splitext(os.path.basename(p))[0]
        video_map[stem] = p

    return video_map


def find_video_path(video_name, video_map):
    video_name = clean_text(video_name).replace("_keypoints", "")

    if video_name in video_map:
        return video_map[video_name]

    for stem, path in video_map.items():
        if video_name == stem:
            return path
        if video_name in stem or stem in video_name:
            return path

    return ""


def repair_csv_path(row):
    old_path = clean_text(row.get("csv_path", ""))

    if os.path.exists(old_path):
        return old_path

    video_name = clean_text(row.get("video_name", "")).replace("_keypoints", "")
    candidate = os.path.join(KEYPOINT_CSV_DIR, f"{video_name}_keypoints.csv")

    if os.path.exists(candidate):
        return candidate

    return old_path


# 1. Load rule-based results
df_rule = pd.read_csv(RULE_BASED_RESULTS_PATH)

print("Rule-based results loaded:", RULE_BASED_RESULTS_PATH)
print("Rule results:", df_rule.shape)

print("\nRule-based distribution:")
print(df_rule["predicted_action"].value_counts(dropna=False))


# 2. Collect already-used videos from batch1 + batch2 + batch3
already_used = set()

for path in OLD_LABEL_FILES:
    if os.path.exists(path):
        df_old = pd.read_csv(path)

        if "video_name" in df_old.columns:
            names = set(df_old["video_name"].astype(str).str.strip())
            already_used.update(names)
            print(f"Loaded used videos from {path}: {len(names)}")
    else:
        print("Not found, skipped:", path)

print("\nTotal already-used videos:", len(already_used))


# 3. Match original video paths
video_files = collect_video_files(VIDEO_ROOT)
video_map = build_video_map(video_files)

print("\nOriginal videos found:", len(video_files))

df_candidates = df_rule.copy()
df_candidates["video_name"] = df_candidates["video_name"].astype(str).str.strip()

if "original_video_path" not in df_candidates.columns:
    df_candidates["original_video_path"] = ""

df_candidates["original_video_path"] = df_candidates.apply(
    lambda row: (
        row["original_video_path"]
        if isinstance(row.get("original_video_path", ""), str)
        and os.path.exists(row.get("original_video_path", ""))
        else find_video_path(row["video_name"], video_map)
    ),
    axis=1,
)

df_candidates["csv_path"] = df_candidates.apply(repair_csv_path, axis=1)

df_candidates["video_exists"] = df_candidates["original_video_path"].apply(
    lambda p: isinstance(p, str) and os.path.exists(p)
)

df_candidates["csv_exists"] = df_candidates["csv_path"].apply(
    lambda p: isinstance(p, str) and os.path.exists(p)
)


# 4. Filter out old videos and invalid paths
df_candidates = df_candidates[
    (~df_candidates["video_name"].isin(already_used)) &
    (df_candidates["video_exists"]) &
    (df_candidates["csv_exists"])
].copy()

print("\nAvailable candidates after removing old labels:")
print("Rows:", len(df_candidates))
print(df_candidates["predicted_action"].value_counts(dropna=False))


# 5. Sample batch4
samples = []

for action, n in TARGET_RULE_ACTIONS.items():
    part = df_candidates[df_candidates["predicted_action"] == action].copy()

    sample_n = min(n, len(part))

    print(f"{action}: available={len(part)}, sampled={sample_n}")

    if sample_n > 0:
        samples.append(part.sample(n=sample_n, random_state=RANDOM_STATE))

if not samples:
    raise RuntimeError("No candidates available for batch4.")

df_batch = pd.concat(samples, ignore_index=True)
df_batch = df_batch.drop_duplicates(subset=["video_name"], keep="first").copy()

# 6. Prepare label columns
df_batch["manual_label"] = ""
df_batch["label_status"] = "unlabeled"
df_batch["notes"] = ""
df_batch["batch_id"] = BATCH_ID

front_cols = [
    "batch_id",
    "video_name",
    "original_video_path",
    "csv_path",
    "predicted_action",
    "manual_label",
    "label_status",
    "notes",
]

other_cols = [c for c in df_batch.columns if c not in front_cols]
df_batch = df_batch[front_cols + other_cols]

df_batch.to_csv(BATCH_SAMPLE_PATH, index=False, encoding="utf-8-sig")

print("\nBatch4 sample saved:")
print(BATCH_SAMPLE_PATH)

print("\nBatch4 size:", len(df_batch))
print("\nBatch4 distribution:")
print(df_batch["predicted_action"].value_counts(dropna=False))

display(df_batch.head(20))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/taekwondo_rule_based_classification_results.csv'

In [ ]:
import os
import hashlib
import subprocess
from pathlib import Path

import cv2
import pandas as pd
from IPython.display import display, clear_output, HTML, Video
import ipywidgets as widgets


# =========================
# 1. Safe Drive mount
# =========================

if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")


# # =========================
# # 2. Paths
# # =========================

# LABEL_PATH = "/content/drive/MyDrive/taekwondo_manual_labeling_sample_120_with_video_path.csv"
# SAVE_PATH = "/content/drive/MyDrive/taekwondo_manual_labels.csv"

# VIDEO_PREVIEW_DIR = Path("/content/drive/MyDrive/taekwondo_labeling_video_previews")
# VIDEO_PREVIEW_DIR.mkdir(exist_ok=True)

# MAX_DURATION_SEC = 20

ROOT = "/content/drive/MyDrive"

BATCH_ID = "batch5"

LABEL_PATH = f"{ROOT}/taekwondo_manual_labeling_{BATCH_ID}_with_video_path.csv"
SAVE_PATH = f"{ROOT}/taekwondo_manual_labels_{BATCH_ID}.csv"

VIDEO_PREVIEW_DIR = Path(f"{ROOT}/taekwondo_labeling_video_previews")
VIDEO_PREVIEW_DIR.mkdir(exist_ok=True)

MAX_DURATION_SEC = 20



# =========================
# 3. Load existing progress first
# =========================

if os.path.exists(SAVE_PATH):
    print("Continue from existing manual labels:")
    print(SAVE_PATH)
    df = pd.read_csv(SAVE_PATH)
else:
    print("Start from labeling template:")
    print(LABEL_PATH)
    df = pd.read_csv(LABEL_PATH)

for col in ["manual_label", "label_status", "notes", "duration_sec", "frame_count", "fps"]:
    if col not in df.columns:
        df[col] = ""

df["manual_label"] = df["manual_label"].fillna("").astype(str)
df["label_status"] = df["label_status"].fillna("unlabeled").astype(str)
df["notes"] = df["notes"].fillna("").astype(str)


# =========================
# 4. Label options
# =========================

labels = [
    "",
    "front_kick",
    "roundhouse_kick",
    "side_kick",
    "axe_kick",
    "back_kick",
    "unknown",
    "punch",
    "exclude",
]


# =========================
# 5. Helper functions
# =========================

def row_is_done(row):
    manual_label = str(row.get("manual_label", "")).strip()
    label_status = str(row.get("label_status", "")).strip().lower()

    return manual_label != "" or label_status == "labeled"


def get_unlabeled_indices():
    return [i for i, row in df.iterrows() if not row_is_done(row)]


def safe_append_note(old_note, new_note):
    old_note = str(old_note).strip()
    if old_note == "" or old_note.lower() == "nan":
        return new_note
    if new_note in old_note:
        return old_note
    return old_note + " | " + new_note


def save_progress():
    df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")


def get_video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)

    cap.release()

    if fps > 0 and frame_count > 0:
        duration_sec = frame_count / fps
    else:
        duration_sec = 0

    return {
        "duration_sec": duration_sec,
        "frame_count": frame_count,
        "fps": fps,
    }


def mark_exclude(idx, reason):
    df.loc[idx, "manual_label"] = "exclude"
    df.loc[idx, "label_status"] = "labeled"
    df.loc[idx, "notes"] = safe_append_note(df.loc[idx, "notes"], reason)
    save_progress()


def make_preview_mp4(video_path):
    video_path = str(video_path)

    stem = Path(video_path).stem
    path_hash = hashlib.md5(video_path.encode("utf-8")).hexdigest()[:8]

    preview_path = VIDEO_PREVIEW_DIR / f"{stem}_{path_hash}_preview.mp4"

    if preview_path.exists() and preview_path.stat().st_size > 0:
        return str(preview_path)

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", "scale=720:-2",
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-preset", "veryfast",
        "-crf", "28",
        "-an",
        str(preview_path),
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if preview_path.exists() and preview_path.stat().st_size > 0:
        return str(preview_path)

    print("Preview conversion failed.")
    print(result.stderr[-1000:])
    return None


def go_next_unlabeled():
    current_idx = state["idx"]

    remaining_after_current = [
        i for i in get_unlabeled_indices()
        if i > current_idx
    ]

    if len(remaining_after_current) > 0:
        state["idx"] = remaining_after_current[0]
        return True

    remaining_all = get_unlabeled_indices()

    if len(remaining_all) > 0:
        state["idx"] = remaining_all[0]
        return True

    return False


def auto_skip_invalid_or_long_videos():
    auto_messages = []

    while True:
        unlabeled = get_unlabeled_indices()

        if len(unlabeled) == 0:
            state["idx"] = 0
            return False, auto_messages

        if state["idx"] not in unlabeled:
            state["idx"] = unlabeled[0]

        i = state["idx"]
        row = df.loc[i]
        video_path = str(row.get("original_video_path", ""))

        if not os.path.exists(video_path):
            mark_exclude(i, "auto_exclude_file_not_found")
            auto_messages.append(f"Auto excluded video {i+1}: file not found")
            continue

        meta = get_video_meta(video_path)

        df.loc[i, "duration_sec"] = meta["duration_sec"]
        df.loc[i, "frame_count"] = meta["frame_count"]
        df.loc[i, "fps"] = meta["fps"]

        if meta["duration_sec"] > MAX_DURATION_SEC:
            reason = f"auto_exclude_video_over_{MAX_DURATION_SEC}_sec"
            mark_exclude(i, reason)
            auto_messages.append(
                f"Auto excluded video {i+1}: duration {meta['duration_sec']:.1f}s > {MAX_DURATION_SEC}s"
            )
            continue

        save_progress()
        return True, auto_messages


# =========================
# 6. Initial state
# =========================

unlabeled_indices = get_unlabeled_indices()

if len(unlabeled_indices) > 0:
    state = {"idx": unlabeled_indices[0]}
    print(f"Continue from first unlabeled video index: {unlabeled_indices[0]}")
else:
    state = {"idx": 0}
    print("All videos are already labeled.")


# =========================
# 7. Widgets
# =========================

label_dropdown = widgets.Dropdown(
    options=labels,
    description="Label:",
    value=""
)

notes_box = widgets.Text(
    description="Notes:",
    placeholder="clear / unclear / bad angle / multiple people"
)

save_button = widgets.Button(description="Save label", button_style="success")
next_button = widgets.Button(description="Next unlabeled")
prev_button = widgets.Button(description="Previous")
exclude_button = widgets.Button(description="Mark exclude", button_style="warning")


# =========================
# 8. Display current video
# =========================

def display_progress_summary():
    display(HTML("<h4>Current labeling summary</h4>"))
    display(df[["manual_label", "label_status"]].value_counts(dropna=False).reset_index(name="count"))


def show_item():
    clear_output(wait=True)

    has_item, auto_messages = auto_skip_invalid_or_long_videos()

    if auto_messages:
        display(HTML(
            "<p style='color:orange;'><b>Auto-skip log:</b><br>" +
            "<br>".join(auto_messages) +
            "</p>"
        ))

    if not has_item:
        display(HTML("<h3>All videos are labeled or automatically excluded.</h3>"))
        display_progress_summary()
        return

    i = state["idx"]
    row = df.loc[i]

    video_path = row.get("original_video_path", "")
    predicted_action = row.get("predicted_action", "")
    video_name = row.get("video_name", "")
    duration_sec = float(row.get("duration_sec", 0) or 0)

    current_label = str(row.get("manual_label", "")).strip()
    label_dropdown.value = current_label if current_label in labels else ""
    notes_box.value = row.get("notes", "")

    labeled_count = sum(row_is_done(row) for _, row in df.iterrows())
    remaining_count = len(get_unlabeled_indices())

    display(HTML(f"""
    <h3>Video {i+1} / {len(df)}</h3>
    <p><b>video_name:</b> {video_name}</p>
    <p><b>rule-based predicted_action:</b> {predicted_action}</p>
    <p><b>current manual_label:</b> {current_label}</p>
    <p><b>duration:</b> {duration_sec:.2f} sec</p>
    <p><b>labeled / done count:</b> {labeled_count}</p>
    <p><b>remaining unlabeled:</b> {remaining_count}</p>
    <p><b>video path:</b> {video_path}</p>
    """))

    if isinstance(video_path, str) and os.path.exists(video_path):
        display(HTML("<p><b>Video preview for manual labeling:</b></p>"))
        display(HTML("<p style='color:gray;'>If this is the first time showing this video, conversion may take a few seconds.</p>"))

        preview_path = make_preview_mp4(video_path)

        if preview_path:
            display(Video(preview_path, embed=True, width=720))
        else:
            display(HTML("<p style='color:red;'>Could not load video preview.</p>"))
    else:
        display(HTML("<p style='color:red;'>Video file not found. It will be excluded automatically.</p>"))

    display(label_dropdown)
    display(notes_box)
    display(widgets.HBox([prev_button, save_button, exclude_button, next_button]))


# =========================
# 9. Button callbacks
# =========================

def on_save_clicked(b):
    if label_dropdown.value == "":
        print("Please choose a label before saving.")
        return

    i = state["idx"]
    video_name = df.loc[i, "video_name"]

    df.loc[i, "manual_label"] = label_dropdown.value
    df.loc[i, "label_status"] = "labeled"
    df.loc[i, "notes"] = notes_box.value

    save_progress()

    print(f"Saved video {i+1}: {video_name} -> {label_dropdown.value}")
    print(f"Saved to: {SAVE_PATH}")

    if go_next_unlabeled():
        show_item()
    else:
        clear_output(wait=True)
        print("All videos are labeled.")
        display_progress_summary()


def on_exclude_clicked(b):
    i = state["idx"]
    video_name = df.loc[i, "video_name"]

    df.loc[i, "manual_label"] = "exclude"
    df.loc[i, "label_status"] = "labeled"
    df.loc[i, "notes"] = notes_box.value if notes_box.value else "manual_exclude"

    save_progress()

    print(f"Excluded video {i+1}: {video_name}")

    if go_next_unlabeled():
        show_item()
    else:
        clear_output(wait=True)
        print("All videos are labeled or excluded.")
        display_progress_summary()


def on_next_clicked(b):
    if go_next_unlabeled():
        show_item()
    else:
        clear_output(wait=True)
        print("No unlabeled videos left.")
        display_progress_summary()


def on_prev_clicked(b):
    if state["idx"] > 0:
        state["idx"] -= 1
    show_item()


save_button.on_click(on_save_clicked)
exclude_button.on_click(on_exclude_clicked)
next_button.on_click(on_next_clicked)
prev_button.on_click(on_prev_clicked)

show_item()

Google Drive is already mounted.
Start from labeling template:
/content/drive/MyDrive/taekwondo_manual_labeling_batch5_with_video_path.csv


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/taekwondo_manual_labeling_batch5_with_video_path.csv'

In [ ]:
import os
import pandas as pd
import shutil

ROOT = "/content/drive/MyDrive"

LABEL_FILES = [
    f"{ROOT}/taekwondo_manual_labels.csv",
    f"{ROOT}/taekwondo_manual_labels_batch2.csv",
    f"{ROOT}/taekwondo_manual_labels_batch3.csv",
    f"{ROOT}/taekwondo_manual_labels_batch4.csv",
]

OUT_PATH = f"{ROOT}/taekwondo_manual_labels_all_batches.csv"
BACKUP_PATH = f"{ROOT}/taekwondo_manual_labels_all_batches_backup.csv"

dfs = []

for path in LABEL_FILES:
    if os.path.exists(path):
        df = pd.read_csv(path)
        df["source_file"] = os.path.basename(path)
        dfs.append(df)
        print("Loaded:", path, "rows:", len(df))
    else:
        print("Missing, skipped:", path)

df_all = pd.concat(dfs, ignore_index=True)

print("\nBefore dedupe:", len(df_all))

df_all = df_all.drop_duplicates(subset=["video_name"], keep="first").copy()

print("After dedupe:", len(df_all))

df_all.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("\nMerged labels saved:")
print(OUT_PATH)

print("\nLabel summary:")
display(
    df_all[["manual_label", "label_status"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

valid_training_labels = [
    "front_kick",
    "roundhouse_kick",
    "side_kick",
    "axe_kick",
    "back_kick",
]

df_train = df_all[
    (df_all["label_status"] == "labeled") &
    (df_all["manual_label"].isin(valid_training_labels))
].copy()

print("\nTraining label distribution:")
print(df_train["manual_label"].value_counts())

print("\nTotal valid training samples:", len(df_train))

Loaded: /content/drive/MyDrive/taekwondo_manual_labels.csv rows: 115
Loaded: /content/drive/MyDrive/taekwondo_manual_labels_batch2.csv rows: 80
Loaded: /content/drive/MyDrive/taekwondo_manual_labels_batch3.csv rows: 180
Loaded: /content/drive/MyDrive/taekwondo_manual_labels_batch4.csv rows: 148

Before dedupe: 523
After dedupe: 523

Merged labels saved:
/content/drive/MyDrive/taekwondo_manual_labels_all_batches.csv

Label summary:


,manual_label,label_status,count
0,back_kick,labeled,95
1,exclude,labeled,93
2,side_kick,labeled,85
3,axe_kick,labeled,84
4,front_kick,labeled,82
5,roundhouse_kick,labeled,79
6,punch,labeled,5



Training label distribution:
manual_label
back_kick          95
side_kick          85
axe_kick           84
front_kick         82
roundhouse_kick    79
Name: count, dtype: int64

Total valid training samples: 425


# Batch 5

In [2]:
# ============================================================
# Cell 1: Create batch5 manual-labeling template
# Goal:
# - Find all remaining unlabeled videos
# - Exclude already labeled batch1-batch4 videos
# - Save batch5 template CSV
# ============================================================

import os
import glob
import pandas as pd
from IPython.display import display

# ============================================================
# 0. Safe Google Drive Mount
# ============================================================

if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")


# ============================================================
# 1. Paths and Settings
# ============================================================

ROOT = "/content/drive/MyDrive"

RULE_BASED_RESULTS_PATH = f"{ROOT}/taekwondo_rule_based_classification_results.csv"

VIDEO_ROOT = f"{ROOT}/taekwondo_videos"
KEYPOINT_CSV_DIR = f"{ROOT}/taekwondo_keypoints_csv_main_actor"

BATCH_ID = "batch5"

BATCH_TEMPLATE_PATH = f"{ROOT}/taekwondo_manual_labeling_{BATCH_ID}_with_video_path.csv"

# If this is False, existing batch5 template will not be overwritten.
FORCE_RECREATE_BATCH5 = False

VIDEO_EXTS = (
    ".mp4", ".MP4",
    ".mov", ".MOV",
    ".avi", ".AVI",
    ".mkv", ".MKV",
)

# Existing completed label files.
# These are used to avoid selecting videos that were already labeled.
OLD_LABEL_FILES = [
    f"{ROOT}/taekwondo_manual_labels.csv",
    f"{ROOT}/taekwondo_manual_labels_batch2.csv",
    f"{ROOT}/taekwondo_manual_labels_batch3.csv",
    f"{ROOT}/taekwondo_manual_labels_batch4.csv",
]

print("Rule-based result path:", RULE_BASED_RESULTS_PATH)
print("Batch5 template output:", BATCH_TEMPLATE_PATH)


# ============================================================
# 2. Helper Functions
# ============================================================

def clean_text(x):
    text = str(x).strip()

    if text.lower() in ["nan", "none", "null"]:
        return ""

    return text


def collect_video_files(video_root):
    video_files = []

    for ext in VIDEO_EXTS:
        video_files.extend(
            glob.glob(os.path.join(video_root, "**", f"*{ext}"), recursive=True)
        )

    return sorted(set(video_files))


def build_video_map(video_files):
    video_map = {}

    for path in video_files:
        stem = os.path.splitext(os.path.basename(path))[0]
        video_map[stem] = path

    return video_map


def find_video_path(video_name, video_map):
    video_name = clean_text(video_name).replace("_keypoints", "")

    if video_name in video_map:
        return video_map[video_name]

    for stem, path in video_map.items():
        if video_name == stem:
            return path

        if video_name in stem or stem in video_name:
            return path

    return ""


def repair_csv_path(row):
    old_path = clean_text(row.get("csv_path", ""))

    if os.path.exists(old_path):
        return old_path

    video_name = clean_text(row.get("video_name", "")).replace("_keypoints", "")
    candidate = os.path.join(KEYPOINT_CSV_DIR, f"{video_name}_keypoints.csv")

    if os.path.exists(candidate):
        return candidate

    return old_path


def is_completed_label_row(row):
    """
    Count only completed rows as already used.
    This prevents an unlabeled template from accidentally excluding videos.
    """
    manual_label = clean_text(row.get("manual_label", ""))
    label = clean_text(row.get("label", ""))
    label_status = clean_text(row.get("label_status", "")).lower()

    if manual_label != "":
        return True

    if label != "":
        return True

    if label_status not in ["", "unlabeled", "pending"]:
        return True

    return False


# ============================================================
# 3. Create Batch5 Template
# ============================================================

if os.path.exists(BATCH_TEMPLATE_PATH) and not FORCE_RECREATE_BATCH5:
    print("Batch5 template already exists. Not overwriting:")
    print(BATCH_TEMPLATE_PATH)

    df_batch5 = pd.read_csv(BATCH_TEMPLATE_PATH)
    print("Existing batch5 size:", len(df_batch5))
    display(df_batch5.head(20))

else:
    if not os.path.exists(RULE_BASED_RESULTS_PATH):
        raise FileNotFoundError(f"Rule-based results not found: {RULE_BASED_RESULTS_PATH}")

    df_rule = pd.read_csv(RULE_BASED_RESULTS_PATH)

    print("Rule-based results loaded:", RULE_BASED_RESULTS_PATH)
    print("Rule result shape:", df_rule.shape)

    if "predicted_action" not in df_rule.columns:
        raise ValueError("Missing required column: predicted_action")

    if "video_name" not in df_rule.columns:
        raise ValueError("Missing required column: video_name")

    print("\nRule-based distribution:")
    print(df_rule["predicted_action"].value_counts(dropna=False))

    # --------------------------------------------------------
    # Collect already labeled videos
    # --------------------------------------------------------

    already_labeled = set()

    for path in OLD_LABEL_FILES:
        if os.path.exists(path):
            df_old = pd.read_csv(path)

            if "video_name" not in df_old.columns:
                print("No video_name column, skipped:", path)
                continue

            completed = df_old[df_old.apply(is_completed_label_row, axis=1)].copy()
            names = set(completed["video_name"].astype(str).str.strip())

            already_labeled.update(names)

            print(f"Loaded completed labels from {path}: {len(names)}")
        else:
            print("Missing, skipped:", path)

    print("\nTotal already labeled videos:", len(already_labeled))

    # --------------------------------------------------------
    # Match original video paths
    # --------------------------------------------------------

    video_files = collect_video_files(VIDEO_ROOT)
    video_map = build_video_map(video_files)

    print("\nOriginal videos found:", len(video_files))

    df_candidates = df_rule.copy()
    df_candidates["video_name"] = df_candidates["video_name"].astype(str).str.strip()

    if "original_video_path" not in df_candidates.columns:
        df_candidates["original_video_path"] = ""

    df_candidates["original_video_path"] = df_candidates.apply(
        lambda row: (
            row["original_video_path"]
            if isinstance(row.get("original_video_path", ""), str)
            and os.path.exists(row.get("original_video_path", ""))
            else find_video_path(row["video_name"], video_map)
        ),
        axis=1,
    )

    df_candidates["csv_path"] = df_candidates.apply(repair_csv_path, axis=1)

    df_candidates["video_exists"] = df_candidates["original_video_path"].apply(
        lambda p: isinstance(p, str) and os.path.exists(p)
    )

    df_candidates["csv_exists"] = df_candidates["csv_path"].apply(
        lambda p: isinstance(p, str) and os.path.exists(p)
    )

    # --------------------------------------------------------
    # Keep only remaining valid candidates
    # --------------------------------------------------------

    df_batch5 = df_candidates[
        (~df_candidates["video_name"].isin(already_labeled)) &
        (df_candidates["video_exists"]) &
        (df_candidates["csv_exists"])
    ].copy()

    df_batch5 = df_batch5.drop_duplicates(subset=["video_name"], keep="first").copy()

    print("\nRemaining candidates for batch5:", len(df_batch5))
    print("\nBatch5 predicted_action distribution:")
    print(df_batch5["predicted_action"].value_counts(dropna=False))

    # --------------------------------------------------------
    # Prepare manual label columns
    # --------------------------------------------------------

    df_batch5["manual_label"] = ""
    df_batch5["label_status"] = "unlabeled"
    df_batch5["notes"] = ""
    df_batch5["batch_id"] = BATCH_ID

    front_cols = [
        "batch_id",
        "video_name",
        "original_video_path",
        "csv_path",
        "predicted_action",
        "manual_label",
        "label_status",
        "notes",
    ]

    other_cols = [c for c in df_batch5.columns if c not in front_cols]
    df_batch5 = df_batch5[front_cols + other_cols]

    df_batch5.to_csv(BATCH_TEMPLATE_PATH, index=False, encoding="utf-8-sig")

    print("\nBatch5 template saved:")
    print(BATCH_TEMPLATE_PATH)

    print("\nBatch5 size:", len(df_batch5))
    display(df_batch5.head(20))

Mounted at /content/drive
Rule-based result path: /content/drive/MyDrive/taekwondo_rule_based_classification_results.csv
Batch5 template output: /content/drive/MyDrive/taekwondo_manual_labeling_batch5_with_video_path.csv
Batch5 template already exists. Not overwriting:
/content/drive/MyDrive/taekwondo_manual_labeling_batch5_with_video_path.csv
Existing batch5 size: 155


,batch_id,video_name,original_video_path,csv_path,predicted_action,manual_label,label_status,notes,right_foot_highest,left_foot_highest,...,left_foot_y_range,right_foot_speed,left_foot_speed,max_foot_speed,body_vertical_range,right_wrist_x_range,left_wrist_x_range,max_wrist_x_range,video_exists,csv_exists
0,batch5,V_20240902_110029_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Unknown,NaN,unlabeled,NaN,0.795045,0.790174,...,0.038703,0.012542,0.012831,0.012831,0.182020,0.086365,0.075858,0.086365,True,True
1,batch5,V_20240902_105504_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Punch / Hand Motion,NaN,unlabeled,NaN,0.811826,0.785650,...,0.036493,0.003579,0.005074,0.005074,0.172210,0.316914,0.110128,0.316914,True,True
2,batch5,V_20240902_104040_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Punch / Hand Motion,NaN,unlabeled,NaN,0.772843,0.737839,...,0.068148,0.011520,0.015524,0.015524,0.172759,0.259550,0.115260,0.259550,True,True
3,batch5,V_20240902_110450_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Unknown,NaN,unlabeled,NaN,0.775159,0.765707,...,0.024937,0.004123,0.003846,0.004123,0.152986,0.215463,0.135199,0.215463,True,True
4,batch5,V_20240902_112308_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Unknown,NaN,unlabeled,NaN,0.760609,0.741514,...,0.038019,0.004179,0.005853,0.005853,0.153252,0.183285,0.069143,0.183285,True,True
5,batch5,VID_20240902_102541,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Unknown,NaN,unlabeled,NaN,0.822943,0.813998,...,0.049972,0.011287,0.013112,0.013112,0.137923,0.181660,0.126300,0.181660,True,True
6,batch5,VID_20240902_103329,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Unknown,NaN,unlabeled,NaN,0.783251,0.785168,...,0.048479,0.079313,0.011694,0.079313,0.141392,0.176734,0.080186,0.176734,True,True
7,batch5,V_20240902_111010_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,NaN,unlabeled,NaN,0.775828,0.207824,...,0.624810,0.008499,0.166450,0.166450,0.152786,0.090476,0.086745,0.090476,True,True
8,batch5,V_20240902_110738_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Front Kick,NaN,unlabeled,NaN,0.432684,0.424917,...,0.417750,0.125795,0.095645,0.125795,0.170025,0.158283,0.102624,0.158283,True,True
9,batch5,V_20240902_110840_ES0,/content/drive/MyDrive/taekwondo_videos/202409...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Unknown,NaN,unlabeled,NaN,0.792280,0.775622,...,0.021441,0.004061,0.004772,0.004772,0.156998,0.135832,0.106593,0.135832,True,True


In [3]:
# ============================================================
# Cell 2: Manual labeling tool for batch5
# Goal:
# - Continue from existing progress if batch5 label file exists
# - Display video preview
# - Save manual labels safely to Google Drive
# ============================================================

import os
import hashlib
import subprocess
from pathlib import Path

import cv2
import pandas as pd
from IPython.display import display, clear_output, HTML, Video
import ipywidgets as widgets

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


# ============================================================
# 0. Safe Google Drive Mount
# ============================================================

if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")


# ============================================================
# 1. Paths and Settings
# ============================================================

ROOT = "/content/drive/MyDrive"

BATCH_ID = "batch5"

LABEL_PATH = f"{ROOT}/taekwondo_manual_labeling_{BATCH_ID}_with_video_path.csv"
SAVE_PATH = f"{ROOT}/taekwondo_manual_labels_{BATCH_ID}.csv"

VIDEO_PREVIEW_DIR = Path(f"{ROOT}/taekwondo_labeling_video_previews")
VIDEO_PREVIEW_DIR.mkdir(exist_ok=True)

MAX_DURATION_SEC = 20

print("Label template:", LABEL_PATH)
print("Save path:", SAVE_PATH)
print("Preview folder:", VIDEO_PREVIEW_DIR)


# ============================================================
# 2. Load Existing Progress or Template
# ============================================================

if os.path.exists(SAVE_PATH):
    print("Continue from existing manual labels:")
    print(SAVE_PATH)
    df = pd.read_csv(SAVE_PATH)

elif os.path.exists(LABEL_PATH):
    print("Start from labeling template:")
    print(LABEL_PATH)
    df = pd.read_csv(LABEL_PATH)

else:
    raise FileNotFoundError(
        f"Neither SAVE_PATH nor LABEL_PATH exists.\n"
        f"SAVE_PATH: {SAVE_PATH}\n"
        f"LABEL_PATH: {LABEL_PATH}\n"
        "Please run Cell 1 first."
    )

required_cols = [
    "manual_label",
    "label_status",
    "notes",
    "duration_sec",
    "frame_count",
    "fps",
]

for col in required_cols:
    if col not in df.columns:
        df[col] = ""

df["manual_label"] = df["manual_label"].fillna("").astype(str)
df["label_status"] = df["label_status"].fillna("unlabeled").astype(str)
df["notes"] = df["notes"].fillna("").astype(str)


# ============================================================
# 3. Label Options
# ============================================================

labels = [
    "",
    "front_kick",
    "roundhouse_kick",
    "side_kick",
    "axe_kick",
    "back_kick",
    "unknown",
    "punch",
    "exclude",
]


# ============================================================
# 4. Helper Functions
# ============================================================

def row_is_done(row):
    manual_label = str(row.get("manual_label", "")).strip()
    label_status = str(row.get("label_status", "")).strip().lower()

    return manual_label != "" or label_status == "labeled"


def get_unlabeled_indices():
    return [i for i, row in df.iterrows() if not row_is_done(row)]


def safe_append_note(old_note, new_note):
    old_note = str(old_note).strip()

    if old_note == "" or old_note.lower() == "nan":
        return new_note

    if new_note in old_note:
        return old_note

    return old_note + " | " + new_note


def save_progress():
    df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")


def get_video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)

    cap.release()

    if fps > 0 and frame_count > 0:
        duration_sec = frame_count / fps
    else:
        duration_sec = 0

    return {
        "duration_sec": duration_sec,
        "frame_count": frame_count,
        "fps": fps,
    }


def mark_exclude(idx, reason):
    df.loc[idx, "manual_label"] = "exclude"
    df.loc[idx, "label_status"] = "labeled"
    df.loc[idx, "notes"] = safe_append_note(df.loc[idx, "notes"], reason)
    save_progress()


def make_preview_mp4(video_path):
    """
    Convert a Drive video into a browser-playable preview.
    The converted preview is cached in Google Drive.
    """
    video_path = str(video_path)

    stem = Path(video_path).stem
    path_hash = hashlib.md5(video_path.encode("utf-8")).hexdigest()[:8]

    preview_path = VIDEO_PREVIEW_DIR / f"{stem}_{path_hash}_preview.mp4"

    if preview_path.exists() and preview_path.stat().st_size > 0:
        return str(preview_path)

    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-vf", "scale=720:-2",
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-preset", "veryfast",
        "-crf", "28",
        "-an",
        str(preview_path),
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if preview_path.exists() and preview_path.stat().st_size > 0:
        return str(preview_path)

    print("Preview conversion failed.")
    print(result.stderr[-1000:])
    return None


def go_next_unlabeled():
    current_idx = state["idx"]

    remaining_after_current = [
        i for i in get_unlabeled_indices()
        if i > current_idx
    ]

    if len(remaining_after_current) > 0:
        state["idx"] = remaining_after_current[0]
        return True

    remaining_all = get_unlabeled_indices()

    if len(remaining_all) > 0:
        state["idx"] = remaining_all[0]
        return True

    return False


def auto_skip_invalid_or_long_videos():
    auto_messages = []

    while True:
        unlabeled = get_unlabeled_indices()

        if len(unlabeled) == 0:
            state["idx"] = 0
            return False, auto_messages

        if state["idx"] not in unlabeled:
            state["idx"] = unlabeled[0]

        i = state["idx"]
        row = df.loc[i]

        video_path = str(row.get("original_video_path", ""))

        if not os.path.exists(video_path):
            mark_exclude(i, "auto_exclude_file_not_found")
            auto_messages.append(f"Auto excluded video {i + 1}: file not found")
            continue

        meta = get_video_meta(video_path)

        df.loc[i, "duration_sec"] = meta["duration_sec"]
        df.loc[i, "frame_count"] = meta["frame_count"]
        df.loc[i, "fps"] = meta["fps"]

        if meta["duration_sec"] > MAX_DURATION_SEC:
            reason = f"auto_exclude_video_over_{MAX_DURATION_SEC}_sec"
            mark_exclude(i, reason)
            auto_messages.append(
                f"Auto excluded video {i + 1}: "
                f"duration {meta['duration_sec']:.1f}s > {MAX_DURATION_SEC}s"
            )
            continue

        save_progress()
        return True, auto_messages


def display_progress_summary():
    display(HTML("<h4>Current labeling summary</h4>"))

    if len(df) == 0:
        print("No rows.")
        return

    display(
        df[["manual_label", "label_status"]]
        .value_counts(dropna=False)
        .reset_index(name="count")
    )


# ============================================================
# 5. Initial State
# ============================================================

unlabeled_indices = get_unlabeled_indices()

if len(unlabeled_indices) > 0:
    state = {"idx": unlabeled_indices[0]}
    print(f"Continue from first unlabeled video index: {unlabeled_indices[0]}")
else:
    state = {"idx": 0}
    print("All videos are already labeled.")


# ============================================================
# 6. Widgets
# ============================================================

label_dropdown = widgets.Dropdown(
    options=labels,
    description="Label:",
    value="",
)

notes_box = widgets.Text(
    description="Notes:",
    placeholder="clear / unclear / bad angle / multiple people",
)

save_button = widgets.Button(description="Save label", button_style="success")
exclude_button = widgets.Button(description="Mark exclude", button_style="warning")
next_button = widgets.Button(description="Next unlabeled")
prev_button = widgets.Button(description="Previous")


# ============================================================
# 7. Display Current Item
# ============================================================

def show_item():
    clear_output(wait=True)

    has_item, auto_messages = auto_skip_invalid_or_long_videos()

    if auto_messages:
        display(HTML(
            "<p style='color:orange;'><b>Auto-skip log:</b><br>" +
            "<br>".join(auto_messages) +
            "</p>"
        ))

    if not has_item:
        display(HTML("<h3>All videos are labeled or automatically excluded.</h3>"))
        display_progress_summary()
        return

    i = state["idx"]
    row = df.loc[i]

    video_path = row.get("original_video_path", "")
    predicted_action = row.get("predicted_action", "")
    video_name = row.get("video_name", "")
    duration_sec = float(row.get("duration_sec", 0) or 0)

    current_label = str(row.get("manual_label", "")).strip()

    label_dropdown.value = current_label if current_label in labels else ""
    notes_box.value = str(row.get("notes", ""))

    labeled_count = sum(row_is_done(row) for _, row in df.iterrows())
    remaining_count = len(get_unlabeled_indices())

    display(HTML(f"""
    <h3>Video {i + 1} / {len(df)}</h3>
    <p><b>video_name:</b> {video_name}</p>
    <p><b>rule-based predicted_action:</b> {predicted_action}</p>
    <p><b>current manual_label:</b> {current_label}</p>
    <p><b>duration:</b> {duration_sec:.2f} sec</p>
    <p><b>labeled / done count:</b> {labeled_count}</p>
    <p><b>remaining unlabeled:</b> {remaining_count}</p>
    <p><b>video path:</b> {video_path}</p>
    """))

    if isinstance(video_path, str) and os.path.exists(video_path):
        display(HTML("<p><b>Video preview for manual labeling:</b></p>"))
        display(HTML("<p style='color:gray;'>If this is the first time showing this video, conversion may take a few seconds.</p>"))

        preview_path = make_preview_mp4(video_path)

        if preview_path:
            display(Video(preview_path, embed=True, width=720))
        else:
            display(HTML("<p style='color:red;'>Could not load video preview.</p>"))
    else:
        display(HTML("<p style='color:red;'>Video file not found. It will be excluded automatically.</p>"))

    display(label_dropdown)
    display(notes_box)
    display(widgets.HBox([prev_button, save_button, exclude_button, next_button]))


# ============================================================
# 8. Button Callbacks
# ============================================================

def on_save_clicked(b):
    if label_dropdown.value == "":
        print("Please choose a label before saving.")
        return

    i = state["idx"]
    video_name = df.loc[i, "video_name"]

    df.loc[i, "manual_label"] = label_dropdown.value
    df.loc[i, "label_status"] = "labeled"
    df.loc[i, "notes"] = notes_box.value

    save_progress()

    print(f"Saved video {i + 1}: {video_name} -> {label_dropdown.value}")
    print(f"Saved to: {SAVE_PATH}")

    if go_next_unlabeled():
        show_item()
    else:
        clear_output(wait=True)
        print("All videos are labeled.")
        display_progress_summary()


def on_exclude_clicked(b):
    i = state["idx"]
    video_name = df.loc[i, "video_name"]

    df.loc[i, "manual_label"] = "exclude"
    df.loc[i, "label_status"] = "labeled"
    df.loc[i, "notes"] = notes_box.value if notes_box.value else "manual_exclude"

    save_progress()

    print(f"Excluded video {i + 1}: {video_name}")

    if go_next_unlabeled():
        show_item()
    else:
        clear_output(wait=True)
        print("All videos are labeled or excluded.")
        display_progress_summary()


def on_next_clicked(b):
    if go_next_unlabeled():
        show_item()
    else:
        clear_output(wait=True)
        print("No unlabeled videos left.")
        display_progress_summary()


def on_prev_clicked(b):
    if state["idx"] > 0:
        state["idx"] -= 1

    show_item()


save_button.on_click(on_save_clicked)
exclude_button.on_click(on_exclude_clicked)
next_button.on_click(on_next_clicked)
prev_button.on_click(on_prev_clicked)


# ============================================================
# 9. Start Tool
# ============================================================

show_item()

,manual_label,label_status,count
0,punch,labeled,94
1,roundhouse_kick,labeled,16
2,axe_kick,labeled,15
3,side_kick,labeled,13
4,front_kick,labeled,10
5,exclude,labeled,6
6,back_kick,labeled,1


In [4]:
import os
import pandas as pd
from IPython.display import display

ROOT = "/content/drive/MyDrive"

LABEL_FILES = [
    f"{ROOT}/taekwondo_manual_labels.csv",
    f"{ROOT}/taekwondo_manual_labels_batch2.csv",
    f"{ROOT}/taekwondo_manual_labels_batch3.csv",
    f"{ROOT}/taekwondo_manual_labels_batch4.csv",
    f"{ROOT}/taekwondo_manual_labels_batch5.csv",
]

OUT_ALL_PATH = f"{ROOT}/taekwondo_manual_labels_all_batches.csv"
OUT_TRAIN_PATH = f"{ROOT}/taekwondo_manual_labels_training_valid.csv"

valid_training_labels = [
    "front_kick",
    "roundhouse_kick",
    "side_kick",
    "axe_kick",
    "back_kick",
]

dfs = []

for order, path in enumerate(LABEL_FILES):
    if os.path.exists(path):
        df_part = pd.read_csv(path)
        df_part["source_file"] = os.path.basename(path)
        df_part["source_order"] = order
        dfs.append(df_part)
        print("Loaded:", path, "rows:", len(df_part))
    else:
        print("Missing, skipped:", path)

if not dfs:
    raise RuntimeError("No label files found.")

df_all = pd.concat(dfs, ignore_index=True)

print("\nBefore cleaning:", len(df_all))

for col in ["video_name", "manual_label", "label_status", "notes"]:
    if col not in df_all.columns:
        df_all[col] = ""

df_all["video_name"] = df_all["video_name"].astype(str).str.strip()
df_all["manual_label"] = df_all["manual_label"].fillna("").astype(str).str.strip()
df_all["label_status"] = df_all["label_status"].fillna("").astype(str).str.strip()

df_all = df_all[df_all["video_name"] != ""].copy()

# If duplicates exist, keep the latest batch version.
df_all = df_all.sort_values(["video_name", "source_order"])
df_all = df_all.drop_duplicates(subset=["video_name"], keep="last").copy()

print("After dedupe:", len(df_all))

df_all.to_csv(OUT_ALL_PATH, index=False, encoding="utf-8-sig")

print("\nMerged all labels saved:")
print(OUT_ALL_PATH)

print("\nFull label summary:")
display(
    df_all[["manual_label", "label_status"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

df_train = df_all[
    (df_all["label_status"].str.lower() == "labeled") &
    (df_all["manual_label"].isin(valid_training_labels))
].copy()

df_train.to_csv(OUT_TRAIN_PATH, index=False, encoding="utf-8-sig")

print("\nValid training labels saved:")
print(OUT_TRAIN_PATH)

print("\nTraining label distribution:")
print(df_train["manual_label"].value_counts())

print("\nTotal valid training samples:", len(df_train))

display(df_train.head(20))

Loaded: /content/drive/MyDrive/taekwondo_manual_labels.csv rows: 115
Loaded: /content/drive/MyDrive/taekwondo_manual_labels_batch2.csv rows: 80
Loaded: /content/drive/MyDrive/taekwondo_manual_labels_batch3.csv rows: 180
Loaded: /content/drive/MyDrive/taekwondo_manual_labels_batch4.csv rows: 148
Loaded: /content/drive/MyDrive/taekwondo_manual_labels_batch5.csv rows: 155

Before cleaning: 678
After dedupe: 678

Merged all labels saved:
/content/drive/MyDrive/taekwondo_manual_labels_all_batches.csv

Full label summary:


,manual_label,label_status,count
0,axe_kick,labeled,99
1,exclude,labeled,99
2,punch,labeled,99
3,side_kick,labeled,98
4,back_kick,labeled,96
5,roundhouse_kick,labeled,95
6,front_kick,labeled,92



Valid training labels saved:
/content/drive/MyDrive/taekwondo_manual_labels_training_valid.csv

Training label distribution:
manual_label
axe_kick           99
side_kick          98
back_kick          96
roundhouse_kick    95
front_kick         92
Name: count, dtype: int64

Total valid training samples: 480


,video_name,original_video_path,csv_path,predicted_action,manual_label,label_status,notes,max_foot_speed,right_foot_y_range,left_foot_y_range,...,right_foot_highest,left_foot_highest,min_foot_y,right_foot_speed,left_foot_speed,right_wrist_x_range,left_wrist_x_range,max_wrist_x_range,video_exists,csv_exists
365,1a6cbd3c40ff4c9d98e7fc9e892ab2a4,/content/drive/MyDrive/taekwondo_videos/junior...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Roundhouse Kick,back_kick,labeled,NaN,0.069551,0.387175,0.052287,...,0.436408,0.772191,0.436408,0.069551,0.014023,0.271654,0.293921,0.293921,True,True
501,530683503931555874,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Roundhouse Kick,side_kick,labeled,NaN,0.153545,0.474463,0.060976,...,0.431177,0.819593,0.431177,0.153545,0.015353,0.080354,0.076958,0.080354,True,True
155,530683505005297923,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Front Kick,side_kick,labeled,NaN,0.118457,0.029520,0.386665,...,0.814854,0.461037,0.461037,0.004395,0.118457,0.078074,0.067808,0.078074,True,True
497,530683507790315621,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Roundhouse Kick,back_kick,labeled,NaN,0.090426,0.390797,0.207885,...,0.464976,0.652804,0.464976,0.090426,0.023485,0.199715,0.136986,0.199715,True,True
652,530683508763918580,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,axe_kick,labeled,NaN,0.176665,0.687208,0.188655,...,0.196686,0.699617,0.196686,0.176665,0.019001,0.110775,0.083738,0.110775,True,True
292,532896955492466898,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Front Kick,front_kick,labeled,NaN,0.127092,0.483043,0.428929,...,0.376296,0.386551,0.376296,0.127092,0.041236,0.191397,0.265044,0.265044,True,True
210,532896972303237242,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Axe Kick / High Kick,roundhouse_kick,labeled,NaN,0.075506,0.615475,0.067377,...,0.227958,0.728658,0.227958,0.075506,0.008825,0.127385,0.122508,0.127385,True,True
176,539968449226211442,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Roundhouse Kick,front_kick,labeled,NaN,0.136987,0.533167,0.457227,...,0.378535,0.424523,0.378535,0.136987,0.102358,0.253454,0.220592,0.253454,True,True
462,539968454460702854,/content/drive/MyDrive/taekwondo_videos/sophom...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Front Kick,side_kick,labeled,NaN,0.114626,0.023381,0.459541,...,0.854584,0.432089,0.432089,0.003591,0.114626,0.137652,0.080843,0.137652,True,True
555,753716469.828646,/content/drive/MyDrive/taekwondo_videos/freshm...,/content/drive/MyDrive/taekwondo_keypoints_csv...,Front Kick,side_kick,labeled,NaN,0.131131,0.453051,0.054810,...,0.468430,0.836465,0.468430,0.131131,0.008715,0.071477,0.076556,0.076556,True,True
